In [1]:
from transformers import EarlyStoppingCallback  # Early stopping callback
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset, Dataset
import warnings
import pandas as pd
import random
import numpy as np
import torch
from transformers import set_seed

# Set random seed
def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Make cudnn use deterministic algorithms
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

# Load custom CSV-format fine-tuning data
def load_custom_dataset(csv_path):
    df = pd.read_csv(csv_path)
    # Ensure the column names are ['Caption', 'SMILES']
    return Dataset.from_pandas(df)

# Data preprocessing, tokenize
def preprocess_function(examples, tokenizer, max_source_length=128, max_target_length=128):
    inputs = examples['Caption']
    targets = examples['SMILES']
    model_inputs = tokenizer(inputs, max_length=max_source_length, truncation=True, padding="max_length")

    labels = tokenizer(targets, max_length=max_target_length, truncation=True, padding="max_length")

    labels["input_ids"] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label] 
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

def main():
    # Paths for model and tokenizer
    model_name_or_path = R"/mnt/slowdisk/ZYH/LLMCP/molt5large_cp2sm"  # Local MolT5-large directory
    train_file = R"/home/zhaoyiheng/LLM/FT_MolT5/dataset/V7/MolGen_Train.csv"  # Training data path
    val_file = R"/home/zhaoyiheng/LLM/FT_MolT5/dataset//V7/MolGen_Evl.csv"  # Validation data path

    tokenizer = T5Tokenizer.from_pretrained(model_name_or_path)
    model = T5ForConditionalGeneration.from_pretrained(model_name_or_path)

    # Load training dataset
    raw_train_datasets = load_custom_dataset(train_file)
    tokenized_train_datasets = raw_train_datasets.map(lambda x: preprocess_function(x, tokenizer), batched=True)

    # Load validation dataset
    raw_val_datasets = load_custom_dataset(val_file)
    tokenized_val_datasets = raw_val_datasets.map(lambda x: preprocess_function(x, tokenizer), batched=True)

    # Training arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir="/mnt/slowdisk/ZYH/LLMCP/CheckPoint/SingMolT5_large_0804",
        evaluation_strategy="epoch",  # Set to "epoch" to evaluate at the end of each epoch
        learning_rate=1e-5,
        per_device_train_batch_size=8, # Due to VRAM limitations, reduce the batch size.
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        save_total_limit=3,
        num_train_epochs=50, # Up to 50 epochs
        predict_with_generate=True,
        logging_dir="/mnt/slowdisk/ZYH/LLMCP/CheckPoint/SingMolT5_large_0804_logs",
        logging_steps=10,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",  # Use validation loss to determine early stopping
        greater_is_better=False,            # Smaller loss is better
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train_datasets,
        eval_dataset=tokenized_val_datasets,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5, early_stopping_threshold=0.02)]  # Stop if no improvement for 3 evaluations
    )

    # Start training
    trainer.train()

if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=FutureWarning, module="accelerate") # ignore FutureWarning to improve output
    set_random_seed(42)
    main()

/usr/local/anaconda3/envs/zyhEnv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-04 17:10:24.830502: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-04 17:10:25.002249: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-04 17:10:25.006791: W tensorflow/compiler/xla/stream_executor/platform

Epoch,Training Loss,Validation Loss
1,1.829500,1.161489
2,1.543300,0.786630
3,0.925100,0.579865
4,0.906700,0.485643
5,0.651700,0.428234
6,0.446700,0.384845
7,0.450000,0.345410
8,0.445900,0.318922
9,0.318600,0.296308
10,0.306600,0.281702


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


# Notice
This warning "There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'] "occurs because the checkpoint stores the token embeddings under shared.weight (the T5 convention of using a single shared embedding). In other words, model.safetensors only contains shared.weight, so the loader reports those two encoder/decoder keys as missing. This is not actually an issue for us because we only use the shared embedding and never rely on encoder.embed_tokens.weight or decoder.embed_tokens.weight separately. For more context see this Hugging Face discussion: https://github.com/huggingface/transformers/issues/27972